# skills

> Know-how an agent can read when it needs it, and extensions a user can drop in.

In [ ]:
#| default_exp skills

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import runpy
from dataclasses import dataclass, field
from pathlib import Path
from fastcore.docments import frontmatter
from shalya.host import host_err

## A skill

A skill is know-how the agent reads on demand: a package that documents itself, or a `SKILL.md` a
person wrote. Two sources, one shape.

`Skill` holds a *way to get* the body rather than the body. Discovering forty skills would otherwise
mean importing forty modules and reading forty files to build a system prompt that names them and
shows none of them. The description is the first paragraph, collapsed to one line, which is the
convention a `SKILL.md` frontmatter already follows.

In [ ]:
#| export
GROUP, EXTRA_MODULES, MAX_SKILL_CHARS = 'pyskills', ('exhash.skill',), 20_000

`GROUP` is the entry-point group installed packages publish under. `EXTRA_MODULES` is the
short list of modules that document themselves without publishing one.

In [ ]:
GROUP, EXTRA_MODULES, MAX_SKILL_CHARS

In [ ]:
test_eq(GROUP, 'pyskills')
test_eq(EXTRA_MODULES, ('exhash.skill',))
test_eq(MAX_SKILL_CHARS, 20_000)

In [ ]:
#| export
def _describe(text, mx=300):
    'A one-line description from a skill body: its first paragraph, collapsed.'
    body = (text or '').strip()
    if not body: return ''
    para = body.split('\n\n', 1)[0]
    one = ' '.join(para.split())
    return one if len(one) <= mx else one[:mx - 1].rstrip() + '…'

@dataclass
class Skill:
    'One skill: how to name it, when it applies, and how to get the whole text.'
    name: str
    source: str                   # 'pyskill' | 'md'
    description: str = ''
    where: str = ''               # module path or file path, shown so a person can go read it
    _text: object = field(default=None, repr=False)

    def text(self):
        'The full skill body, clipped. Never raises: a broken skill reports itself as one.'
        try: t = self._text() if callable(self._text) else (self._text or '')
        except Exception as e: return f'could not read skill {self.name}: {host_err(e)}'
        t = str(t)
        return t if len(t) <= MAX_SKILL_CHARS else t[:MAX_SKILL_CHARS] + f'\n…[{len(t)-MAX_SKILL_CHARS} more chars]'

    def dict(self): return {'name': self.name, 'source': self.source,'description': self.description, 'where': self.where}

In [ ]:
_describe('''Search the web and read results into durable memory.

Use it when a question needs pages nobody has read yet.''')

In [ ]:
test_eq(_describe(''), '')
test_eq(_describe('one\ntwo\n\nthree'), 'one two')
test_eq(len(_describe('x ' * 400)), 300)

The body is fetched once, when something asks for it, and clipped. A skill that cannot be read
reports itself as one rather than raising into a turn.

In [ ]:
s = Skill('nbdev', 'md', 'Develop nbdev projects.', 'nbs/SKILL.md', _text=lambda: 'the whole body')
def explodes(): raise FileNotFoundError('SKILL.md')
broken = Skill('gone', 'md', _text=explodes)
s.text(), broken.text()

In [ ]:
test_eq(s.text(), 'the whole body')
assert broken.text().startswith('could not read skill gone'), broken.text()
big = Skill('big', 'md', _text='x' * (MAX_SKILL_CHARS + 500))
test_eq(len(big.text()), MAX_SKILL_CHARS + len('\n…[500 more chars]'))
test_eq(s.dict()['where'], 'nbs/SKILL.md')

## Where skills come from

Installed packages publish theirs through the `pyskills` entry-point group. Nothing imports a module
to list it, except that a description has to come from the docstring, and there is no way to read one
without importing.

In [ ]:
#| export
def _mod_skill(name, modpath):
    "A `Skill` for a module, without importing it until someone asks for the body."
    def load():
        from importlib import import_module
        return import_module(modpath).__doc__ or ''
    # the description needs the docstring, and there is no way to read one without importing
    try:
        from importlib import import_module
        doc = import_module(modpath).__doc__ or ''
    except Exception: return None
    if not doc.strip(): return None
    return Skill(name=name, source='pyskill', description=_describe(doc), where=modpath, _text=load)

def _pyskills():
    "Every module published under the `pyskills` entry-point group, plus the known stragglers."
    out, seen = [], set()
    try:
        from importlib.metadata import entry_points
        eps = list(entry_points(group=GROUP))
    except Exception:
        eps = []
    for ep in eps:
        mod = getattr(ep, 'value', None) or ep.name
        if mod in seen: continue
        seen.add(mod)
        if (s := _mod_skill(ep.name.split('.')[-1] or ep.name, mod)): out.append(s)
    for mod in EXTRA_MODULES:
        if mod in seen: continue
        seen.add(mod)
        if (s := _mod_skill(mod.split('.')[0], mod)): out.append(s)
    return out

`exhash` ships a skill module, and it is a dependency here, so discovery finds a real one.

In [ ]:
found = _pyskills()
[(s.name, s.source, s.where) for s in found]

In [ ]:
assert any(s.where == 'exhash.skill' for s in found), [s.where for s in found]
ex = next(s for s in found if s.where == 'exhash.skill')
assert ex.description, 'a pyskill with no description should have been dropped'
assert len(ex.text()) > 100, len(ex.text())

Files come from a search path in increasing precedence. Personal habits first, then a project's,
so a repository can override what a person set up for themselves.

In [ ]:
#| export
def skill_dirs(roots=(), cfg=None):
    "Where SKILL.md files are looked for, in increasing precedence: user first. A project can override."
    from pathlib import Path
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'skills')
    ds.append(Path.home()/'.agents'/'skills')
    for r in roots: ds += [Path(r)/'.leela'/'skills', Path(r)/'.agents'/'skills']
    return ds

def _md_skills(d):
    "Skills in one directory, following the Agent Skills layout: `<name>/SKILL.md`."
    from pathlib import Path
    d = Path(d)
    if not d.is_dir(): return []
    out = []
    for p in sorted(d.iterdir()):
        if not p.is_dir(): continue
        f = p/'SKILL.md'
        if not f.exists(): continue
        try: raw = f.read_text(encoding='utf-8')
        except Exception: continue
        meta, body = frontmatter(raw)
        out.append(Skill(name=meta.get('name') or p.name, source='md',
                         description=meta.get('description') or _describe(body),
                         where=str(f), _text=body))
    return out

def discover(roots=(), cfg=None, extra=()):
    "Every skill available to this agent. Pyskills, then `skill_dirs`, then `extra`. Later winning."
    by_name = {}
    for s in _pyskills(): by_name[s.name] = s
    for d in skill_dirs(roots, cfg):
        for s in _md_skills(d): by_name[s.name] = s
    for s in extra or (): by_name[s.name] = s
    return sorted(by_name.values(), key=lambda s: s.name)

In [ ]:
[str(p) for p in skill_dirs(roots=['/proj'], cfg='/home/k/.config/leela')]

In [ ]:
ds = [str(p) for p in skill_dirs(roots=['/proj'], cfg='/cfg')]
test_eq(ds[0], '/cfg/skills')
test_eq(ds[-1], '/proj/.agents/skills')

Written out, a skill directory follows the Agent Skills layout: `<name>/SKILL.md`, with
frontmatter supplying the name and description when it wants to.

In [ ]:
tmp = Path(tempfile.mkdtemp())
d = tmp/'.agents'/'skills'/'notebook-tests'
d.mkdir(parents=True)
(d/'SKILL.md').write_text('''---
name: notebook-tests
description: Write executable test cells in the notebook that owns the code.
---

Put the readable case in the notebook and the bulk assertions in `tests/`.
''')
plain = tmp/'.agents'/'skills'/'plain'
plain.mkdir()
(plain/'SKILL.md').write_text('Say what changed.\n\nAnd nothing else.\n')
[(s.name, s.description) for s in _md_skills(tmp/'.agents'/'skills')]

In [ ]:
got = {s.name: s for s in _md_skills(tmp/'.agents'/'skills')}
test_eq(sorted(got), ['notebook-tests', 'plain'])
test_eq(got['plain'].description, 'Say what changed.')     # no frontmatter: the first paragraph
assert 'bulk assertions' in got['notebook-tests'].text()
test_eq(got['notebook-tests'].source, 'md')

`discover` merges every source, with a later one winning a name clash.

In [ ]:
skills = discover(roots=[tmp])
[s.name for s in skills]

In [ ]:
names = [s.name for s in skills]
assert 'notebook-tests' in names and 'exhash' in names, names
test_eq(names, sorted(names))
mine = discover(roots=[tmp], extra=[Skill('plain', 'ext', 'Mine wins.')])
test_eq(next(s for s in mine if s.name == 'plain').description, 'Mine wins.')

## The index

What goes in the system prompt is names and clipped descriptions, never bodies. One verbose skill
cannot crowd out the rest.

In [ ]:
#| export
SKILL_DESC_MAX = 160   # per skill. One verbose description cannot crowd out the rest

def _clip_desc(s, n=SKILL_DESC_MAX):
    "One line, clipped at a word boundary: in the index a description only has to be pickable."
    s = ' '.join(str(s).split())
    if len(s) <= n: return s
    cut = s.rfind(' ', 0, n)
    return s[:cut if cut > 0 else n].rstrip(' .,;:\u2014-') + '\u2026'

def skill_index(skills):
    "The block that goes in the system prompt: names and clipped descriptions, never bodies."
    if not skills: return ''
    rows = '\n'.join(f'- `{s.name}` -- {_clip_desc(s.description)}' for s in skills)
    return ('\n\n## Skills\n\nKnow-how available to you. Read one with `read_skill(name)` when its '
            'description matches what you are about to do, *before* you do it -- several of these '
            'describe tools already installed in this environment, so the code they discuss is '
            'also searchable with `search_code`.\n\n' + rows)

def find(skills, name):
    "A skill by exact name, then unique prefix, then unique substring. Ambiguity is None, not a guess."
    if not name: return None
    n = name.strip().lower()
    if (exact := [s for s in skills if s.name.lower() == n]): return exact[0]
    for pred in (lambda s: s.name.lower().startswith(n), lambda s: n in s.name.lower()):
        if len(hits := [s for s in skills if pred(s)]) == 1: return hits[0]
    return None

In [ ]:
print(skill_index([s for s in skills if s.name == 'notebook-tests']))

In [ ]:
ix = skill_index([s for s in skills if s.name == 'notebook-tests'])
assert '`notebook-tests`' in ix and 'read_skill' in ix
assert 'Put the readable case' not in ix, 'the index must not carry a body'
test_eq(skill_index([]), '')
test_eq(len(_clip_desc('word ' * 60)), SKILL_DESC_MAX)

A name resolves exactly, then by unique prefix, then by unique substring. Ambiguity answers None
rather than guessing, and the tool that called it then lists what there is.

In [ ]:
find(skills, 'notebook-tests').name, find(skills, 'notebook').name, find(skills, 'nope')

In [ ]:
test_eq(find(skills, 'notebook-tests').name, 'notebook-tests')
test_eq(find(skills, 'notebook').name, 'notebook-tests')     # unique prefix
test_eq(find(skills, 'tests').name, 'notebook-tests')        # unique substring
test_eq(find(skills, 'nope'), None)
two = [Skill('edit', 'md'), Skill('editor', 'md')]
test_eq(find(two, 'edit').name, 'edit')                      # exact beats prefix
test_eq(find(two, 'edi'), None)                              # ambiguous prefix

## Extensions

An extension is a Python file a user drops in a directory. `Registry` is everything it may add, and
it is deliberately not a route to anything else: no backend, no engine, no history.

In [ ]:
#| export
EVENTS = ('before_turn', 'after_turn', 'before_tool', 'after_tool', 'compact', 'approval')

The lifecycle events an extension may hook. Unknown names are refused rather than silently never
firing, which is the whole reason this is a closed list.

In [ ]:
EVENTS

In [ ]:
test_eq(EVENTS, ('before_turn', 'after_turn', 'before_tool', 'after_tool', 'compact', 'approval'))
test_eq(len(set(EVENTS)), len(EVENTS))

In [ ]:
#| export
class Registry:
    "What `setup(ext)` is handed: everything an extension may add, and no route to a backend's internals."

    def __init__(self, host=None, agent=None):
        self.host, self.agent = host, agent
        self.tools, self.skills, self.commands = [], [], {}
        self.hooks = {e: [] for e in EVENTS}
        self.approve = None
        self.notes = []          # one line per extension: loaded, or why not


    def tool(self, f):
        "Add a tool, as a decorator: a plain function with type hints and the docstring the model reads."
        self.tools.append(f)
        return f

    def skill(self, name, text, description=''):
        "Add a skill the discovery pass would not find. A file, a string, anything callable."
        s = Skill(name=name, source='ext', description=description or _describe(text if isinstance(text, str) else ''),
                  where='extension', _text=text)
        self.skills.append(s)
        return s

    def command(self, name, fn, help=''):
        "Add a slash command. `fn(agent, arg)` returns text for the frontend to show."
        self.commands[name.lstrip('/')] = (fn, help)
        return fn

    def on(self, event, fn):
        "Hook a harness lifecycle event. Unknown event names are an error, not a silent no-op."
        if event not in EVENTS: raise KeyError(f'unknown event {event!r}; known: {", ".join(EVENTS)}')
        self.hooks[event].append(fn)
        return fn

    def approval(self, fn):
        "Replace the approval policy wholesale. The last extension to call this wins."
        self.approve = fn
        return fn


    def fire(self, event, *args, **kw):
        "Run every hook for `event`, swallowing failures. Returns how many ran cleanly."
        n = 0
        for f in self.hooks.get(event, ()):
            try: f(*args, **kw); n += 1
            except Exception as e: self.notes.append(f'{event} hook failed: {host_err(e)}')
        return n

def ext_dirs(roots=(), cfg=None, project=False):
    "Where extensions are looked for. Project directories only when explicitly allowed."
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'extensions')
    if project:
        for r in roots: ds.append(Path(r)/'.leela'/'extensions')
    return ds

def load(reg, roots=(), cfg=None, project=False, paths=()):
    "Run every extension found, calling its `setup(reg)`. A file with no `setup` is left alone."
    files = []
    for d in ext_dirs(roots, cfg, project):
        if Path(d).is_dir(): files += sorted(p for p in Path(d).glob('*.py') if not p.name.startswith('_'))
    for p in paths or ():
        p = Path(p)
        files += sorted(p.glob('*.py')) if p.is_dir() else [p]
    for f in files:
        try: ns = runpy.run_path(str(f))
        except Exception as e:
            reg.notes.append(f'{f.name}: failed to load ({host_err(e)})')
            continue
        fn = ns.get('setup')
        if not callable(fn):
            reg.notes.append(f'{f.name}: loaded, no setup()')
            continue
        before = (len(reg.tools), len(reg.skills), len(reg.commands))
        try: fn(reg)
        except Exception as e:
            reg.notes.append(f'{f.name}: setup() failed ({host_err(e)})')
            continue
        d = [n - b for n, b in zip((len(reg.tools), len(reg.skills), len(reg.commands)), before)]
        reg.notes.append(f'{f.name}: {d[0]} tool(s), {d[1]} skill(s), {d[2]} command(s)')
    return reg

A real extension, with a real `setup(reg)`, loaded off disk.

In [ ]:
extdir = tmp/'extensions'
extdir.mkdir()
(extdir/'wc.py').write_text('''
def setup(reg):
    @reg.tool
    def word_count(path: str) -> str:
        "How many words are in a file."
        return str(len(open(path).read().split()))
    reg.command('wc', lambda agent, arg: 'counted ' + arg, help='count words')
    reg.skill('house-style', 'Dense lines. Short names.', 'How code is written here.')
''')
(extdir/'helpers.py').write_text('SHARED = 1\n')
(extdir/'boom.py').write_text('raise RuntimeError("no")\n')
reg = load(Registry(), cfg=tmp)
reg.notes

In [ ]:
test_eq([t.__name__ for t in reg.tools], ['word_count'])
test_eq(list(reg.commands), ['wc'])
test_eq([s.name for s in reg.skills], ['house-style'])
test_eq(reg.skills[0].text(), 'Dense lines. Short names.')
test_eq(reg.tools[0](str(extdir/'helpers.py')), '3')          # the tool really runs
assert any('helpers.py: loaded, no setup()' in n for n in reg.notes), reg.notes
assert any(n.startswith('boom.py: failed to load') for n in reg.notes), reg.notes

Hooking an event that does not exist is an error rather than a silent no-op: a misspelled hook
that never fires is worse than one that refuses to register.

In [ ]:
test_fail(lambda: reg.on('before_lunch', print), contains='unknown event')

Firing is fail-soft in the other direction. A hook that raises is recorded and the turn carries on,
because an extension is not allowed to end someone's session.

In [ ]:
fired = []
reg.on('before_turn', lambda **kw: 1/0)
reg.on('before_turn', lambda **kw: fired.append(kw))
reg.fire('before_turn', prompt='hello'), fired, reg.notes[-1]

In [ ]:
test_eq(fired, [{'prompt': 'hello'}])
assert reg.notes[-1].startswith('before_turn hook failed'), reg.notes[-1]

An extension may replace the approval policy wholesale. The last one to call this wins, which is
why it is a single slot rather than a list of hooks.

In [ ]:
@reg.approval
def approve_reads(tool, **kw): return not tool.startswith('write_')
reg.approve, reg.approve('view_file'), reg.approve('write_file')

In [ ]:
test_eq(reg.approve.__name__, 'approve_reads')
test_eq(reg.approve('view_file'), True)
test_eq(reg.approve('write_file'), False)
test_eq(Registry().approve, None)                         # nothing registered approves everything

Project directories are read only when a caller says so. A repository that can register a tool by
being cloned is a repository that can register a tool by being cloned.

In [ ]:
test_eq([str(p) for p in ext_dirs(roots=['/proj'], cfg='/cfg')], ['/cfg/extensions'])
test_eq([str(p) for p in ext_dirs(roots=['/proj'], cfg='/cfg', project=True)][-1],
        '/proj/.leela/extensions')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()